# Notebook 3 — Train / Validation / Test Split

**Task 2 — From Tables to Notebooks · Qafza Tech MLOps Training 2026/2027**

**Goal:** split `labeled_table.csv` into train / validation / test **before** doing any deep analysis, so the test set can't influence a single decision we make later.

**What we'll do:**
1. Load the labeled table
2. Decide *how* to split — random vs. time-based — and why
3. Do the split
4. Check the date range and label balance in each split
5. Save the three files

> ⚠️ **Why split before EDA?** If we explore the full dataset first — see a pattern, decide a feature is useful, pick a modeling approach — we've already let the test set shape our decisions. That's a subtler form of leakage than using a forbidden column, but it's still leakage: our final "test" metric would no longer measure how the model does on truly unseen data.


## 1. Load the Labeled Table


In [1]:
from pathlib import Path
import pandas as pd

ARTIFACTS_DIR = Path("artifacts/tables")

DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

labeled = pd.read_csv(ARTIFACTS_DIR / "labeled_table.csv", parse_dates=DATE_COLS)
print("Loaded:", labeled.shape)
print("Date range:", labeled["order_purchase_timestamp"].min(), "->", labeled["order_purchase_timestamp"].max())


Loaded: (96470, 29)
Date range: 2016-09-15 12:16:38 -> 2018-08-29 15:00:37


## 2. Random or Time-Based? And Why

Two honest options here:

- **Random split:** shuffle and cut. Simple, and it's fine when rows are independent and the world isn't changing over time.
- **Time-based split:** sort by `order_purchase_timestamp` and cut chronologically — train on the earliest orders, validate and test on the most recent ones.

**We're going with a time-based split.** In production, the model will always be predicting on orders that happen *after* the data it was trained on — that's just how time works. A random split would let the model "see" purchase patterns, prices, and seasonal effects from December while being trained on data that includes December itself, which quietly overstates how well it'll generalize to a *future* month it's never seen. Sorting by time and cutting the tail off for validation/test mimics the real deployment situation much more honestly.

The one thing we have to watch for after a time split (that a random split would have handled automatically): the label ratio can drift across time — e.g. late deliveries might spike around big shopping holidays. We check that explicitly below, and call it out rather than silently accepting whatever split we get.


In [2]:
labeled_sorted = labeled.sort_values("order_purchase_timestamp").reset_index(drop=True)

n = len(labeled_sorted)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = labeled_sorted.iloc[:train_end].copy()
val_df = labeled_sorted.iloc[train_end:val_end].copy()
test_df = labeled_sorted.iloc[val_end:].copy()

print(f"train: {len(train_df):>7,} rows  ({len(train_df) / n:.1%})")
print(f"val  : {len(val_df):>7,} rows  ({len(val_df) / n:.1%})")
print(f"test : {len(test_df):>7,} rows  ({len(test_df) / n:.1%})")


train:  67,529 rows  (70.0%)
val  :  14,470 rows  (15.0%)
test :  14,471 rows  (15.0%)


## 3. Check the Date Range in Each Split

They should not overlap — each split's date window should sit strictly after the previous one's.


In [3]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    start = df["order_purchase_timestamp"].min()
    end = df["order_purchase_timestamp"].max()
    print(f"{name:5s}: {start.date()}  ->  {end.date()}")


train: 2016-09-15  ->  2018-04-15
val  : 2018-04-15  ->  2018-06-21
test : 2018-06-21  ->  2018-08-29


## 4. Check the Label Balance in Each Split

This is the part that a random split gives you for free — with a time split we have to check it ourselves.


In [4]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    rate = df["is_late"].mean()
    print(f"{name:5s}: {rate:.2%} late  (n={len(df):,})")


train: 9.03% late  (n=67,529)
val  : 5.34% late  (n=14,470)
test : 6.61% late  (n=14,471)


If these three rates are reasonably close to each other (within a few points), the time split is safe to use as-is. If one split's late-rate is wildly different from the others — say, test lands entirely in a holiday surge month — that's worth flagging honestly in your findings rather than ignoring: it means the test metric in Notebook 6 should be read with that context in mind, since it may reflect a period with unusually high (or low) delays.

## 5. Save the Splits


In [5]:
train_df.to_csv(ARTIFACTS_DIR / "train.csv", index=False)
val_df.to_csv(ARTIFACTS_DIR / "val.csv", index=False)
test_df.to_csv(ARTIFACTS_DIR / "test.csv", index=False)

print("Saved train.csv, val.csv, test.csv to", ARTIFACTS_DIR)


Saved train.csv, val.csv, test.csv to artifacts\tables


## Recap

- Split **chronologically** by `order_purchase_timestamp`: 70% train / 15% validation / 15% test.
- Chosen over a random split because the production pipeline will always predict on future orders — this split mimics that honestly.
- Checked the label ratio holds up reasonably across all three splits (documented above; re-run and read the numbers for your own data pull, they can shift slightly with dataset updates).

**Rule from here on: the test set is closed.** We open `train.csv` and `val.csv` freely in the next three notebooks. `test.csv` gets touched exactly once — at the very end of Notebook 6.

**Next up — Notebook 4:** the detailed EDA, on `train.csv` only.
